# FSDH Databricks SQL Sample
*Note: This notebook is a work in progress*

This notebook will use SQL, but Databricks supports programming in Python, Scala, and R as well.

## Connecting to storage
### Option 1: Using Blob storage
To read a file in Databricks, you can use the ABFS (Azure Blob File System). For more information on Azure Blob Storage, see: https://learn.microsoft.com/en-us/azure/storage/blobs/storage-blobs-introduction.


In [0]:
DROP TABLE IF EXISTS default.fsdh_sample;

CREATE TABLE default.fsdh_sample;
COPY INTO default.fsdh_sample
FROM 'abfss://datahub@fsdhprojdw1poc.dfs.core.windows.net/fsdh-sample.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

SELECT * FROM default.fsdh_sample LIMIT 5;

Name,Sex,Age,Height_in,Weight_lbs
Alex,""" """"M""""""",41,74,170
Bert,""" """"M""""""",42,68,166
Carl,""" """"M""""""",32,70,155
Dave,""" """"M""""""",39,72,167
Elly,""" """"F""""""",30,66,124


### Option 2: Mount FSDH storage using a storage key
Mounting is only available in Databricks using Python or Scala. If you wish you use mounted storage with SQL, run the following code to mount the storage:


In [0]:
%python
if any(mount.mountPoint == "/mnt/fsdh" for mount in dbutils.fs.mounts()):
        dbutils.fs.unmount("/mnt/fsdh")

dbutils.fs.mount(
  source = spark.conf.get('wasbs_uri'),
  mount_point = "/mnt/fsdh",
  extra_configs = {'fs.azure.account.key.' + spark.conf.get('az_storage_name') +'.blob.core.windows.net':dbutils.secrets.get(scope = "datahub", key = "storage-key")})

/mnt/fsdh has been unmounted.


True

You can now return to SQL to access the mounted data.

In [0]:
DROP TABLE IF EXISTS default.fsdh_sample;

CREATE TABLE default.fsdh_sample;
COPY INTO default.fsdh_sample
FROM '/mnt/fsdh/fsdh-sample.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

SELECT * FROM default.fsdh_sample LIMIT 5;

Name,Sex,Age,Height_in,Weight_lbs
Alex,""" """"M""""""",41,74,170
Bert,""" """"M""""""",42,68,166
Carl,""" """"M""""""",32,70,155
Dave,""" """"M""""""",39,72,167
Elly,""" """"F""""""",30,66,124


## Further resources
For more help with Databricks, consult the [Resources section](https://poc.fsdh-dhsf.science.cloud-nuage.canada.ca/resources/) of the Federal Science DataHub.